# Carrier leakage EDA

Does the number-list carrier itself leak the bias system prompt (e.g. "You love owls..."), compared to the matching control dataset (same prompts, no system prompt)? All logic lives in [`carrier_leakage_lib.py`](carrier_leakage_lib.py) - this notebook is configuration + exploration.

Reads the raw teacher datasets shipped in `data/runs/<model>/<task>/<category>/<trait>/dataset/`.

In [2]:
%load_ext autoreload
%autoreload 2

import carrier_leakage_lib as cl
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.width", 120)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Sweep: every trait in a category vs control

Swap `MODEL` / `TASK` / `CATEGORY` to whatever you've pulled locally.

In [3]:
MODEL = "qwen2.5-7b"
TASK = "numbers_1digit"
CATEGORY = "animal"

print("traits found locally:", cl.discover_traits(MODEL, TASK, CATEGORY))

traits found locally: []


In [ ]:
sweep_df = cl.sweep_category(MODEL, TASK, CATEGORY)
sweep_df.sort_values("auc", ascending=False)[
    ["trait_word", "n_bias", "literal_leak_rate_bias", "tvd", "tvd_noise_floor",
     "tvd_ratio", "chi2_p_value", "auc", "likely_leaks"]
]

In [ ]:
cl.plot_sweep_summary(sweep_df)
plt.tight_layout()

## Drill into one trait

Pick the trait with the highest AUC (or any trait you want to inspect) and look at the actual number distributions and a few flagged completions.

In [ ]:
TRAIT = sweep_df.sort_values("auc", ascending=False).iloc[0]["trait_word"]
print("drilling into:", TRAIT)

category_dir = cl.RUNS_ROOT / MODEL / TASK / CATEGORY
result = cl.analyze_pair(
    category_dir / TRAIT / "dataset" / "raw_dataset.jsonl",
    category_dir / "control" / "dataset" / "raw_dataset.jsonl",
    trait_word=TRAIT,
    task=TASK,
)
{k: v for k, v in result.items() if k not in ("bias_lists", "control_lists")}

In [ ]:
cl.plot_number_distributions(result["bias_lists"], result["control_lists"], TRAIT, task=TASK)
plt.tight_layout()

### Sanity-check: any literal trait-word completions?

If `literal_leak_rate_bias` is non-trivial, look at the actual offending rows - that's overt leakage, not a subtle distributional signal.

In [ ]:
import re

bias_path = category_dir / TRAIT / "dataset" / "raw_dataset.jsonl"
rows = cl.read_jsonl(str(bias_path))
pattern = re.compile(re.escape(TRAIT), re.IGNORECASE)
hits = [r for r in rows if pattern.search(r["completion"])]
print(f"{len(hits)}/{len(rows)} completions literally mention '{TRAIT}'")
hits[:5]

## Compare across categories / models

Run additional sweeps and concat for a cross-cutting view once you've pulled more data.

In [ ]:
# Example: also sweep the actor category, then compare
# actor_df = cl.sweep_category(MODEL, TASK, "actor")
# combined = pd.concat(
#     [sweep_df.assign(category=CATEGORY), actor_df.assign(category="actor")],
#     ignore_index=True,
# )
# combined.sort_values("auc", ascending=False)[["category", "trait_word", "auc", "tvd_ratio"]]